In [1]:
print("Hello, World!")

Hello, World!


# ABC Credit: two-wheeler loan approval, data cleaning pipeline

**Goal:** turn `data/training_data.csv` into model-ready data, so that the **same code** can later clean the hidden hold-out test set and live chatbot inputs without breaking.

**Design principle (for the panel):** we split the cleaning into two layers.

| Layer | What it does | Learns from data? | Applied to |
|---|---|---|---|
| **1. Rule-based cleaning** (`basic_clean`) | Fixes formats, enforces business rules, creates features | No, fixed rules only | Train, test, chatbot |
| **2. Learned preprocessing** (scikit-learn `Pipeline`) | Fills gaps, caps extremes, encodes categories | Yes, **from training rows only** | Fitted once on train, then reused |

Some steps only make sense when we know the answer, such as dropping duplicates or rows with no Decision. Those are applied to **training data only**. We never drop an applicant from the test set.

In [2]:
# ---- Setup ----
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import chi2_contingency
from IPython.display import display

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42          # fixed seed: same results every run
DATA_PATH = "../data/training_data.csv"

## 1. Load the raw file, safely

We read **every column as text** first (`dtype=str`). This matters because:
- **Pincode** is a postal code, not a number. Reading it as a number would turn `560068` into `560068.0` and treat it as "bigger/smaller", which is meaningless.
- **Age** contains a blank space `" "` in one row, which would otherwise make pandas guess the type wrongly.

We convert to numbers ourselves, under our own rules.

In [3]:
raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
print("Shape:", raw.shape)
raw.head()

Shape: (108423, 17)


,ID,Qualifications,Employment_Type,Age,Pincode,Gender,Product_Code,Loan_Amount,LTV,Resident_Type,Net_salary,Final_Tier,Make_Code,Model_Description,Model_Variant,PAST_LOANS_ACTIVE,Decision
0,1,SSC,AGR,47,502110,Male,MO,59300,69.76,O,39000,07 Rural,MOPEDS,TVS XL 100 HD ITS BSVI - WINNER EDITION,XL 100CC,PAST_LOANS_ACTIVE,APPROVED
1,2,UG,SAL,30,503002,Male,MC,153839,84.99,O,25000,05 Urban,APACHE,APACHE 200 4V 2CH - R MODE BSVI,RTR 200CC,PAST_LOANS_ACTIVE,APPROVED
2,3,GRAD,NREGI,24,500090,Male,SC,77958,64.97,R,35000,04 Metro,JUPITER,JUPITER ZX BSVI - AOL - DRUM,110 CC,PAST_LOANS_ACTIVE,APPROVED
3,4,UG,SAL,44,501141,Male,SC,99921,84.68,O,25000,07 Rural,JUPITER,JUPITER 125 BSVI - DISC,125 CC DISC,PAST_LOANS_ACTIVE,APPROVED
4,5,SSC,SAL,41,509334,Male,SC,96482,75.38,O,35000,07 Rural,JUPITER,JUPITER 125 BSVI DISC SXC U566,125CC DISC,PAST_LOANS_ACTIVE,APPROVED


## 2. Verify the known issues against the data

We check each issue in the brief instead of assuming it.

In [4]:
def to_num(s):
    # Text -> number. Blanks / junk become NaN instead of crashing.
    return pd.to_numeric(s.str.strip(), errors="coerce")

stripped = raw.apply(lambda c: c.str.strip())
age, sal, ltv = to_num(raw["Age"]), to_num(raw["Net_salary"]), to_num(raw["LTV"])

checks = {
    "1a. Fully empty rows":                      (stripped == "").all(axis=1).sum(),
    "1b. Rows with missing Decision":            (stripped["Decision"] == "").sum(),
    "2.  Age values that are a blank space ' '": (raw["Age"] == " ").sum(),
    "3.  Age > 60 (outside policy)":             (age > 60).sum(),
    "3b. Max age":                               age.max(),
    "4a. Net_salary == 0":                       (sal == 0).sum(),
    "4b. Net_salary < 5,000":                    (sal < 5000).sum(),
    "4c. Max Net_salary":                        sal.max(),
    "6.  Exact duplicate rows (ignoring ID)":    raw.drop(columns="ID").duplicated().sum(),
    "7a. Pincode unique values":                 raw["Pincode"].nunique(),
    "7b. Pincode not exactly 6 digits":          (~stripped["Pincode"].str.fullmatch(r"\d{6}")).sum(),
    "8.  % DECLINED":                            round(100 * (stripped["Decision"] == "DECLINED").mean(), 2),
    # ---- issues NOT in the brief, found during profiling ----
    "NEW: LTV == 0 (impossible)":                (ltv == 0).sum(),
    "NEW: Garbled text 'â€“' in Model_Description": raw["Model_Description"].str.contains("â€“").sum(),
    "NEW: Model_Variant spelling variants ('125CC' vs '125 CC')": raw["Model_Variant"].str.contains(r"\dCC").sum(),
}
pd.Series(checks, name="value").to_frame()

,value
1a. Fully empty rows,1.00
1b. Rows with missing Decision,1.00
2. Age values that are a blank space ' ',1.00
3. Age > 60 (outside policy),1105.00
3b. Max age,69.00
4a. Net_salary == 0,626.00
"4b. Net_salary < 5,000",887.00
4c. Max Net_salary,980000.00
6. Exact duplicate rows (ignoring ID),13.00
7a. Pincode unique values,14000.00


In [5]:
# 5. Missing values per column (blank text counts as missing)
(stripped == "").sum().loc[lambda s: s > 0].to_frame("missing_count")

,missing_count
ID,1
Qualifications,26
Employment_Type,28
Age,20
Pincode,1
Gender,1
Product_Code,1
Loan_Amount,1
LTV,1
Resident_Type,20


In [6]:
# Who reports zero salary? (decides whether 0 is an error or a real answer)
pd.crosstab(raw["Employment_Type"], sal == 0).rename(columns={False: "salary>0", True: "salary==0"})

Net_salary,salary>0,salary==0
Employment_Type,,
,9,19
AGR,21050,0
NONEARNMEM,2,6
NPP,505,0
NREGI,14102,0
PEN,135,0
SAL,33479,0
SEP,38363,0
STU,152,601


### What the verification tells us

| # | Brief said | Data shows | Decision |
|---|---|---|---|
| 1 | 1 empty row + 1 missing Decision | They are the **same row**: the last row is fully empty | Drop it (training only) |
| 2 | Age has a `" "` | Confirmed (1 row) | Treat as missing |
| 3 | ~1,100 Age > 60, max 69 | 1,105 rows, max 69. **1,068 of them were APPROVED** | **Keep** + add a flag (see below) |
| 4 | Salary min 0, ~890 < 5,000 | 626 zeros, and **601 of them are students (STU)** | Zero is valid for non-earners; tiny values for earners are errors |
| 5 | 20–28 missing per column | Confirmed. Missing values cluster in the same ~80 rows | Impute + flag |
| 6 | 12 duplicates | **13** (ignoring ID) | Drop extra copies (training only) |
| 7 | ≥1 Pincode not 6 digits | Only the empty row; all others are valid 6-digit codes. The bad value only shows up if Pincode is read as a number | Validate anyway, for the test set |
| 8 | 94.7% / 5.3% | Confirmed | Class weights, not resampling |
| new | — | 65 rows with LTV = 0 | Impossible (loan > 0), so treat as missing |
| new | — | Garbled dash `â€“` and `125CC` vs `125 CC` | Standardise text |

**Why keep applicants over 60?** The business approved 1,068 of them, so the policy is clearly not a hard cut-off in practice (for example, a co-applicant may be involved). If we deleted them, the model would never learn about older applicants, yet they will appear in the hold-out set. Policy enforcement belongs in a **rule layer in the chatbot**, before the model, not in deleting history. We add an `age_above_policy` flag so the model can still treat them differently.

## 3. Layer 1: rule-based cleaning (`basic_clean`)

This function uses **only fixed rules**. It learns nothing from the data, so it gives the same result on train, test, or a single chatbot applicant. It never drops rows.

In [7]:
# ---- Business constants (from the data dictionary) ----
AGE_POLICY_MIN, AGE_POLICY_MAX = 18, 60
AGE_PLAUSIBLE_MAX = 100           # beyond this it is a typing error, not an old applicant
NON_EARNER_TYPES = {"STU", "NONEARNMEM"}   # students / non-earning members: salary 0 is a real answer
MIN_PLAUSIBLE_SALARY = 1000       # Rs/month; below this, an earner's salary is a typo (e.g. 1, 25, 100)

# Products that map to a single make / variant (verified: MO -> 16,476 rows all MOPEDS; EB -> 3,452 rows all TVS)
PRODUCT_TO_MAKE = {"MO": ("MOPEDS", "XL 100 CC"), "EB": ("TVS", "EBIKE")}

TIER_ORDER = {  # ordinal: 1 = largest city ... 7 = rural
    "01": 1, "02": 2, "03": 3, "04": 4, "05": 5, "06": 6, "07": 7,
}

CAT_COLS = ["Qualifications", "Employment_Type", "Gender", "Product_Code", "Resident_Type",
            "Make_Code", "Model_Description", "Model_Variant", "PAST_LOANS_ACTIVE"]


def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    # Deterministic, row-by-row cleaning. Safe to run on ANY new data (no learning).
    df = df.copy()

    # 1) Text hygiene: trim spaces, turn blanks into proper missing values
    for c in df.columns:
        if df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype("string").str.strip().replace("", pd.NA)

    # 2) Fix garbled characters (an en-dash saved in the wrong encoding) and case
    for c in ["Model_Description", "Model_Variant", "Make_Code"]:
        df[c] = df[c].str.replace("â€“", "-", regex=False).str.upper()
    # '125CC DISC' and '125 CC DISC' are the same bike variant
    df["Model_Variant"] = df["Model_Variant"].str.replace(r"(\d+)\s*CC", r"\1 CC", regex=True)

    # 2b) Missing bike make/variant: moped (MO) and e-bike (EB) each have exactly ONE make and
    #     variant in the data, so a blank can be filled with certainty. Others stay missing.
    for prod, (make, variant) in PRODUCT_TO_MAKE.items():
        is_prod = (df["Product_Code"] == prod).fillna(False).astype(bool)
        df.loc[is_prod & df["Make_Code"].isna(), "Make_Code"] = make
        df.loc[is_prod & df["Model_Variant"].isna(), "Model_Variant"] = variant

    # 3) Numbers: convert text to numbers; anything unreadable becomes missing
    for c in ["Age", "Loan_Amount", "LTV", "Net_salary"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)   # plain float: NaN, not pd.NA

    # 4) Age: impossible values -> missing; outside-policy ages are KEPT but flagged
    df.loc[(df["Age"] < AGE_POLICY_MIN) | (df["Age"] > AGE_PLAUSIBLE_MAX), "Age"] = np.nan
    df["age_above_policy"] = (df["Age"] > AGE_POLICY_MAX).astype(int)

    # 5) LTV must be strictly between 0 and 100 (a loan of Rs X cannot be 0% of the bike value)
    df["ltv_invalid"] = (~df["LTV"].between(0, 100, inclusive="right") & df["LTV"].notna()).astype(int)
    df.loc[df["ltv_invalid"] == 1, "LTV"] = np.nan

    # 6) Salary: 0 is genuine for non-earners; tiny values for earners are data-entry errors
    is_non_earner = df["Employment_Type"].isin(NON_EARNER_TYPES).fillna(False).astype(bool)
    bad_salary = (df["Net_salary"] < MIN_PLAUSIBLE_SALARY) & ~is_non_earner
    df["salary_invalid"] = bad_salary.fillna(False).astype(int)
    df.loc[bad_salary.fillna(False), "Net_salary"] = np.nan
    df.loc[df["Net_salary"] < 0, "Net_salary"] = np.nan
    df["is_non_earner"] = is_non_earner.astype(int)
    # (very high salaries are capped later, using limits learned from training data)

    # 7) Pincode: keep as TEXT; validate format (6 digits, cannot start with 0)
    pin = df["Pincode"].astype("string").str.replace(r"\.0$", "", regex=True)
    df["Pincode"] = pin.where(pin.str.fullmatch(r"[1-9]\d{5}").fillna(False).astype(bool), pd.NA)
    # Postal hierarchy: first digit = zone, first 3 digits = sorting district.
    # These generalise better than the 14,000 raw pincodes and survive unseen pincodes.
    df["pin_zone"] = df["Pincode"].str[:1]
    df["pin_district"] = df["Pincode"].str[:3]

    # 8) Final_Tier: ordinal rank 1..7. Unknown/new labels -> missing (imputed later)
    df["tier_rank"] = df["Final_Tier"].str[:2].map(TIER_ORDER).astype(float)

    # 9) Business features
    vehicle_value = df["Loan_Amount"] / (df["LTV"] / 100)
    df["vehicle_value"] = vehicle_value
    monthly = df["Net_salary"].where(df["Net_salary"] > 0)
    df["loan_to_annual_income"] = df["Loan_Amount"] / (12 * monthly)   # NaN for zero income
    df["n_missing_inputs"] = df[["Qualifications", "Employment_Type", "Age", "Resident_Type",
                                  "Net_salary", "Final_Tier"]].isna().sum(axis=1)

    # 10) Final type tidy-up: categoricals as plain object with np.nan (what scikit-learn expects)
    for c in CAT_COLS + ["Pincode", "pin_zone", "pin_district", "Final_Tier"]:
        df[c] = df[c].astype(object).where(df[c].notna(), np.nan)
    return df

## 4. Training-only row filters

These steps use the **answer** (Decision) or describe the training file itself, so they are **never** applied to the test set.

In [8]:
def training_row_filter(df: pd.DataFrame) -> pd.DataFrame:
    n0 = len(df)
    feature_cols = [c for c in raw.columns if c != "ID"]   # original columns only (derived flags are never empty)

    # a) Fully empty rows: no information at all
    df = df[~df[feature_cols].isna().all(axis=1)]
    n1 = len(df)
    # b) No Decision: we cannot learn from an applicant whose outcome is unknown
    df = df[df["Decision"].isin(["APPROVED", "DECLINED"])]
    n2 = len(df)
    # c) Exact duplicates (ignoring ID): the same application entered twice.
    #    Keeping both would over-weight it and leak between train and validation.
    df = df.drop_duplicates(subset=[c for c in raw.columns if c != "ID"])
    n3 = len(df)
    # d) Label conflicts: identical applicant details, opposite decisions. Kept (only a handful),
    #    but flagged: they show some decisions depend on things not in our data (label noise).
    inputs = [c for c in raw.columns if c not in ("ID", "Decision")]
    n_outcomes = df.groupby(inputs, dropna=False)["Decision"].transform("nunique")
    df = df.assign(label_conflict=(n_outcomes > 1).astype(int))

    print(f"Start rows             : {n0:,}")
    print(f"- fully empty          : {n0 - n1:,}")
    print(f"- missing Decision     : {n1 - n2:,}")
    print(f"- duplicates           : {n2 - n3:,}")
    print(f"Rows kept for training : {n3:,}")
    print(f"  of which label-conflict rows (kept, flagged): {df['label_conflict'].sum():,}")
    return df.reset_index(drop=True)


clean = training_row_filter(basic_clean(raw))
clean.head()

Start rows             : 108,423
- fully empty          : 1
- missing Decision     : 0
- duplicates           : 13
Rows kept for training : 108,409
  of which label-conflict rows (kept, flagged): 6


,ID,Qualifications,Employment_Type,Age,Pincode,Gender,Product_Code,Loan_Amount,LTV,Resident_Type,Net_salary,Final_Tier,Make_Code,Model_Description,Model_Variant,PAST_LOANS_ACTIVE,Decision,age_above_policy,ltv_invalid,salary_invalid,is_non_earner,pin_zone,pin_district,tier_rank,vehicle_value,loan_to_annual_income,n_missing_inputs,label_conflict
0,1,SSC,AGR,47.0,502110,Male,MO,59300.0,69.76,O,39000.0,07 Rural,MOPEDS,TVS XL 100 HD ITS BSVI - WINNER EDITION,XL 100 CC,PAST_LOANS_ACTIVE,APPROVED,0,0,0,0,5,502,7.0,85005.733945,0.126709,0,0
1,2,UG,SAL,30.0,503002,Male,MC,153839.0,84.99,O,25000.0,05 Urban,APACHE,APACHE 200 4V 2CH - R MODE BSVI,RTR 200 CC,PAST_LOANS_ACTIVE,APPROVED,0,0,0,0,5,503,5.0,181008.353924,0.512797,0,0
2,3,GRAD,NREGI,24.0,500090,Male,SC,77958.0,64.97,R,35000.0,04 Metro,JUPITER,JUPITER ZX BSVI - AOL - DRUM,110 CC,PAST_LOANS_ACTIVE,APPROVED,0,0,0,0,5,500,4.0,119990.764968,0.185614,0,0
3,4,UG,SAL,44.0,501141,Male,SC,99921.0,84.68,O,25000.0,07 Rural,JUPITER,JUPITER 125 BSVI - DISC,125 CC DISC,PAST_LOANS_ACTIVE,APPROVED,0,0,0,0,5,501,7.0,117998.346717,0.333070,0,0
4,5,SSC,SAL,41.0,509334,Male,SC,96482.0,75.38,O,35000.0,07 Rural,JUPITER,JUPITER 125 BSVI DISC SXC U566,125 CC DISC,PAST_LOANS_ACTIVE,APPROVED,0,0,0,0,5,509,7.0,127994.162908,0.229719,0,0


In [9]:
# Sanity checks after Layer 1: every rule should hold
assert clean["Age"].dropna().between(18, 100).all()
assert clean["LTV"].dropna().between(0, 100, inclusive="right").all()
assert clean["Pincode"].dropna().str.fullmatch(r"[1-9]\d{5}").all()
assert not clean.drop(columns="ID").duplicated().any()
print("All Layer-1 checks passed.")
clean[["age_above_policy", "ltv_invalid", "salary_invalid", "is_non_earner"]].sum().to_frame("rows_flagged")

All Layer-1 checks passed.


,rows_flagged
age_above_policy,1105
ltv_invalid,65
salary_invalid,45
is_non_earner,761


## 4b. Who gets declined? Decline rate by segment

These tables are for understanding the business, not for the model. The overall decline rate is about 5.3%.

In [10]:
def decline_table(col):
    t = (clean.groupby(col, dropna=False)["Decision"]
              .agg(applications="size", declined=lambda s: (s == "DECLINED").sum()))
    t["share_of_rows"] = t["applications"] / len(clean)
    t["decline_rate"] = t["declined"] / t["applications"]
    return t.sort_values("applications", ascending=False)

for col in ["Final_Tier", "Employment_Type", "Product_Code", "PAST_LOANS_ACTIVE",
            "Make_Code", "Qualifications", "Resident_Type", "age_above_policy", "Gender"]:
    print(f"\n=== Decline rate by {col}")
    display(decline_table(col).style.format({"share_of_rows": "{:.1%}", "decline_rate": "{:.2%}"}))


=== Decline rate by Final_Tier


,applications,declined,share_of_rows,decline_rate
Final_Tier,,,,
07 Rural,72384,3902,66.8%,5.39%
06 Semi-Urban,21508,1135,19.8%,5.28%
05 Urban,6012,308,5.5%,5.12%
02 Megapolis (B),3661,175,3.4%,4.78%
04 Metro,2859,116,2.6%,4.06%
03 Large Metro,1282,44,1.2%,3.43%
01 Megapolis (A),681,18,0.6%,2.64%
nan,22,0,0.0%,0.00%



=== Decline rate by Employment_Type


,applications,declined,share_of_rows,decline_rate
Employment_Type,,,,
SEP,38362,1982,35.4%,5.17%
SAL,33476,1817,30.9%,5.43%
AGR,21045,1062,19.4%,5.05%
NREGI,14100,787,13.0%,5.58%
STU,753,14,0.7%,1.86%
NPP,505,35,0.5%,6.93%
PEN,135,0,0.1%,0.00%
nan,25,1,0.0%,4.00%
NONEARNMEM,8,0,0.0%,0.00%



=== Decline rate by Product_Code


,applications,declined,share_of_rows,decline_rate
Product_Code,,,,
MC,51664,2930,47.7%,5.67%
SC,36816,1937,34.0%,5.26%
MO,16476,746,15.2%,4.53%
EB,3453,85,3.2%,2.46%


... (8 outputs left out)


=== Decline rate by age_above_policy


,applications,declined,share_of_rows,decline_rate
age_above_policy,,,,
0,107304,5661,99.0%,5.28%
1,1105,37,1.0%,3.35%



=== Decline rate by Gender


,applications,declined,share_of_rows,decline_rate
Gender,,,,
Male,83023,4356,76.6%,5.25%
Female,25386,1342,23.4%,5.29%


### Gender is excluded from the model

The test below asks: *is the decline rate for women really different from men, or is the gap just chance?* A p-value well above 0.05 means **no real difference**.

Gender therefore adds no predictive value, and using it creates **fair-lending and reputational risk** (RBI Fair Practices Code). We keep the column in the data for fairness audits, so we can check the model's decisions are equal across genders, but the model never sees it.

In [11]:
gender_tab = pd.crosstab(clean["Gender"], clean["Decision"])
chi2, p_value, _, _ = chi2_contingency(gender_tab)
display(gender_tab.assign(decline_rate=lambda t: (t["DECLINED"] / t.sum(axis=1)).map("{:.2%}".format)))
print(f"Chi-square p-value = {p_value:.2f} -> {'no significant difference' if p_value > 0.05 else 'significant'}")

Decision,APPROVED,DECLINED,decline_rate
Gender,,,
Female,24044,1342,5.29%
Male,78667,4356,5.25%


Chi-square p-value = 0.82 -> no significant difference


### Label conflicts: same applicant details, opposite decisions

In [12]:
inputs = [c for c in raw.columns if c not in ("ID", "Decision")]
clean.loc[clean["label_conflict"] == 1, ["ID"] + inputs + ["Decision"]].sort_values(inputs[:6])

,ID,Qualifications,Employment_Type,Age,Pincode,Gender,Product_Code,Loan_Amount,LTV,Resident_Type,Net_salary,Final_Tier,Make_Code,Model_Description,Model_Variant,PAST_LOANS_ACTIVE,Decision
26383,26385,GRAD,SAL,24.0,203205,Male,MC,101295.0,87.17,O,28000.0,07 Rural,RAIDER,TVS RAIDER DISC BSVI,RAIDER,PAST_LOANS_ACTIVE,APPROVED
103284,103297,GRAD,SAL,24.0,203205,Male,MC,101295.0,87.17,O,28000.0,07 Rural,RAIDER,TVS RAIDER DISC BSVI,RAIDER,PAST_LOANS_ACTIVE,DECLINED
80847,80855,OTHERS,AGR,30.0,457993,Male,MC,103822.0,67.89,O,25000.0,07 Rural,APACHE,APACHE RTR 160 2V RM DISC BSVI,160 DISC,NO_PAST_LOANS,DECLINED
80858,80866,OTHERS,AGR,30.0,457993,Male,MC,103822.0,67.89,O,25000.0,07 Rural,APACHE,APACHE RTR 160 2V RM DISC BSVI,160 DISC,NO_PAST_LOANS,APPROVED
63525,63532,NaN,NaN,NaN,452001,Female,MO,47000.0,67.87,NaN,NaN,06 Semi-Urban,MOPEDS,TVS XL 100 HD KS BSVI,XL 100 CC,PAST_LOANS_ACTIVE,DECLINED
77772,77779,NaN,NaN,NaN,452001,Female,MO,47000.0,67.87,NaN,NaN,06 Semi-Urban,MOPEDS,TVS XL 100 HD KS BSVI,XL 100 CC,PAST_LOANS_ACTIVE,APPROVED


These rows are identical on every input, yet one was approved and one declined. No model can get both right, so they show **irreducible noise**: some decisions depend on information we don't have, such as a bureau score or the credit officer's judgment. There are only a few, so we keep them; `label_conflict` is **not** a model input. This is also a point for the panel: adding a bureau score to the chatbot would likely help the model more than any extra cleaning.

## 5. Split before learning anything

We hold out 20% as a **validation set** *before* calculating any medians, caps or encodings. If we computed them on all the data, information from the validation rows would leak into training and our accuracy would look better than it really is on the hidden test set.

`stratify` keeps the 5.3% decline rate identical in both parts.

**Target:** `1 = DECLINED`. The rare outcome is the one the business most needs to catch.

In [13]:
TARGET = "Decision"
y = (clean[TARGET] == "DECLINED").astype(int)
X = clean.drop(columns=[TARGET, "ID", "label_conflict"])   # label_conflict uses the answer -> never a feature

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print(f"Train: {len(X_train):,} rows, decline rate {y_train.mean():.2%}")
print(f"Val  : {len(X_val):,} rows, decline rate {y_val.mean():.2%}")

Train: 86,727 rows, decline rate 5.26%
Val  : 21,682 rows, decline rate 5.26%


## 6a. Smarter, business-aware filling (learned from training rows only)

Two gaps can be filled better than with a plain median:

| Gap | Simple fill | Better fill used here | Why |
|---|---|---|---|
| **LTV = 0 / missing** (65 rows) | Median LTV (~81%) | `Loan_Amount ÷ typical price of that bike model` | We know the loan amount and the bike, so we can estimate the bike's price and rebuild the real LTV. LTV is a key risk driver, so it should be as accurate as possible. |
| **Final_Tier missing** (22 rows) | Median tier (Rural) | Most common tier among pincodes with the same **first 3 digits** | Neighbouring pincodes are in the same city or district, so they share a tier. |

Fallbacks keep it unbreakable on new data: bike model → make → product → overall; pincode district → zone → median.

In [14]:
class DomainImputer(BaseEstimator, TransformerMixin):
    # Learns lookup tables on TRAINING rows; applies them to any data.

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        ok = X["LTV"].notna() & X["vehicle_value"].notna()
        vv = X.loc[ok, "vehicle_value"]
        # typical (median) bike price at each level of detail
        self.vv_by_model_ = vv.groupby(X.loc[ok, "Model_Description"]).median()
        self.vv_by_make_ = vv.groupby(X.loc[ok, "Make_Code"]).median()
        self.vv_by_product_ = vv.groupby(X.loc[ok, "Product_Code"]).median()
        self.vv_overall_ = vv.median()
        # most common tier per 3-digit pincode district, and per 1-digit zone
        t = X.dropna(subset=["tier_rank"])
        mode = lambda s: s.mode().iloc[0]
        self.tier_by_district_ = t.dropna(subset=["pin_district"]).groupby("pin_district")["tier_rank"].agg(mode)
        self.tier_by_zone_ = t.dropna(subset=["pin_zone"]).groupby("pin_zone")["tier_rank"].agg(mode)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        # --- rebuild LTV from loan amount and the bike's typical price ---
        need = X["LTV"].isna() & X["Loan_Amount"].notna()
        price = (X["Model_Description"].map(self.vv_by_model_)
                 .fillna(X["Make_Code"].map(self.vv_by_make_))
                 .fillna(X["Product_Code"].map(self.vv_by_product_))
                 .fillna(self.vv_overall_))
        X.loc[need, "LTV"] = (100 * X.loc[need, "Loan_Amount"] / price[need]).clip(1, 100)
        X["vehicle_value"] = X["Loan_Amount"] / (X["LTV"] / 100)
        monthly = X["Net_salary"].where(X["Net_salary"] > 0)
        X["loan_to_annual_income"] = X["Loan_Amount"] / (12 * monthly)
        # --- fill tier from neighbouring pincodes ---
        miss = X["tier_rank"].isna()
        guess = X["pin_district"].map(self.tier_by_district_).fillna(X["pin_zone"].map(self.tier_by_zone_))
        X.loc[miss, "tier_rank"] = guess[miss]
        X["tier_imputed"] = miss.astype(int)
        return X


domain = DomainImputer().fit(X_train)
check = domain.transform(X_train)
rebuilt = X_train["LTV"].isna() & X_train["Loan_Amount"].notna()
print(f"LTV rebuilt for {rebuilt.sum()} training rows; range "
      f"{check.loc[rebuilt, 'LTV'].min():.1f}% to {check.loc[rebuilt, 'LTV'].max():.1f}%, "
      f"median {check.loc[rebuilt, 'LTV'].median():.1f}%")
print(f"Tier filled for {check['tier_imputed'].sum()} rows; still missing: {check['tier_rank'].isna().sum()}")

LTV rebuilt for 51 training rows; range 34.7% to 96.4%, median 80.8%
Tier filled for 20 rows; still missing: 0


## 6b. Layer 2: learned preprocessing (fitted on training rows only)

| Column group | Treatment | Why |
|---|---|---|
| Money columns (Loan, Salary, vehicle value, loan-to-income) | Cap at 0.5th / 99.5th percentile → fill gaps with **median** (+ "was missing" flag) → scale | Rs 9.8 lakh/month on a Rs 90k bike loan is an outlier that would distort the model. The median is robust to extremes. |
| Age, LTV | Fill gaps with median (+ flag) → scale. **No capping** | Already bounded by business rules in Layer 1. Capping Age would hide the over-60 group. |
| `tier_rank` (ordinal 1–7) | Already filled from pincode in 6a; median as a last resort | Keeps the natural order Megapolis → Rural. Unseen tier labels become missing in Layer 1, then get filled. |
| **Gender** | **Not used** | No signal and a fair-lending risk (section 4b). |
| Low-cardinality categories | Fill gaps with `"Missing"` → one-hot; rare (<30) or **unseen** categories go into one "infrequent" bucket | A new bike model in the test set won't crash the model. |
| Pincode, pin district, Model_Description | **Target encoding** (average decline rate, smoothed, cross-fitted) | 14,000 pincodes is too many for one-hot. Smoothing pulls rare pincodes toward the overall average, and unseen pincodes get the overall average. |

In [15]:
class QuantileCapper(BaseEstimator, TransformerMixin):
    # Caps each numeric column at percentiles LEARNED FROM TRAINING DATA (winsorising).
    def __init__(self, lower=0.005, upper=0.995):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lo_ = X.quantile(self.lower)
        self.hi_ = X.quantile(self.upper)
        return self

    def transform(self, X):
        return pd.DataFrame(X).clip(self.lo_, self.hi_, axis=1).to_numpy()

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features)


# Money columns have long right tails -> cap them. Age and LTV are already bounded by business rules.
MONEY_COLS = ["Loan_Amount", "Net_salary", "vehicle_value", "loan_to_annual_income"]
BOUNDED_COLS = ["Age", "LTV"]
FLAG_COLS = ["age_above_policy", "ltv_invalid", "salary_invalid", "is_non_earner", "n_missing_inputs",
             "tier_imputed"]
ORD_COLS = ["tier_rank"]
# NOTE: Gender deliberately excluded (see section 4b)
OHE_COLS = ["Qualifications", "Employment_Type", "Product_Code", "Resident_Type",
            "Make_Code", "Model_Variant", "PAST_LOANS_ACTIVE", "pin_zone"]
HIGH_CARD_COLS = ["Pincode", "pin_district", "Model_Description"]

money_pipe = Pipeline([
    ("cap", QuantileCapper()),
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
bounded_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
ordinal_pipe = Pipeline([("impute", SimpleImputer(strategy="median"))])
ohe_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30,
                          sparse_output=False)),
])
high_card_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("te", TargetEncoder(target_type="binary", smooth="auto",
                         cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))),
])

preprocess = ColumnTransformer([
    ("money", money_pipe, MONEY_COLS),
    ("bounded", bounded_pipe, BOUNDED_COLS),
    ("flags", "passthrough", FLAG_COLS),
    ("ord", ordinal_pipe, ORD_COLS),
    ("ohe", ohe_pipe, OHE_COLS),
    ("te", high_card_pipe, HIGH_CARD_COLS),
], remainder="drop", verbose_feature_names_out=True)

# Full learned pipeline = business-aware filling (6a) + generic preprocessing (6b)
full_prep = Pipeline([("domain", DomainImputer()), ("prep", preprocess)])
Xt_train = full_prep.fit_transform(X_train, y_train)
Xt_val = full_prep.transform(X_val)
feature_names = full_prep.named_steps["prep"].get_feature_names_out()
assert not any("Gender" in f for f in feature_names), "Gender must not be a model input"
print("Model-ready matrix:", Xt_train.shape, "| any NaN left?", np.isnan(Xt_train).any())
cap = preprocess.named_transformers_["money"].named_steps["cap"]
pd.DataFrame({"cap_low": cap.lo_, "cap_high": cap.hi_}).round(2)

Model-ready matrix: (86727, 81) | any NaN left? False


,cap_low,cap_high
Loan_Amount,32400.00,152005.92
Net_salary,0.00,100000.00
vehicle_value,52509.61,188516.86
loan_to_annual_income,0.06,0.60


## 7. Class imbalance (94.7% / 5.3%)

- We **do not** oversample or undersample in the cleaning step. Resampling changes the data itself and is easy to leak into validation.
- Instead, models use `class_weight="balanced"`, so each decline counts about 18× an approval during training.
- **Accuracy is misleading.** A model that approves everyone scores 94.7%. We report **ROC-AUC** and **PR-AUC** (precision-recall, which focuses on the declines). The approve/decline cut-off is a business choice made later, based on the cost of a bad loan vs. a lost customer.

Quick baselines, only to prove the cleaned data flows end-to-end:

In [16]:
models = {
    "Logistic regression": LogisticRegression(class_weight="balanced", max_iter=2000),
    "Gradient boosting": HistGradientBoostingClassifier(class_weight="balanced",
                                                        random_state=RANDOM_STATE),
}
rows = []
for name, model in models.items():
    model.fit(Xt_train, y_train)
    p = model.predict_proba(Xt_val)[:, 1]
    rows.append({"model": name,
                 "ROC-AUC": roc_auc_score(y_val, p),
                 "PR-AUC": average_precision_score(y_val, p),
                 "PR-AUC of random guessing": y_val.mean()})
pd.DataFrame(rows).round(3)

,model,ROC-AUC,PR-AUC,PR-AUC of random guessing
0,Logistic regression,0.780,0.170,0.053
1,Gradient boosting,0.784,0.174,0.053


## 8. Robustness test: does it survive messy, unseen data?

We build fake "hold-out / chatbot" applicants that break every rule: unseen bike model, unseen tier, unseen pincode, blank age, 5-digit pincode, zero salary for a salaried person, LTV = 0, and a completely empty applicant. The full pipeline must return a prediction for **every** row without crashing.

In [17]:
nasty = pd.DataFrame([
    {"ID": "T1", "Qualifications": "PHD", "Employment_Type": "SAL", "Age": " ", "Pincode": "56006",
     "Gender": "Male", "Product_Code": "MC", "Loan_Amount": "95000", "LTV": "0", "Resident_Type": "O",
     "Net_salary": "0", "Final_Tier": "08 Village", "Make_Code": "NEWBIKE",
     "Model_Description": "TVS NEW MODEL 2027", "Model_Variant": "999CC", "PAST_LOANS_ACTIVE": "NO_PAST_LOANS"},
    {"ID": "T2", "Qualifications": "GRAD", "Employment_Type": "STU", "Age": "19", "Pincode": "999999",
     "Gender": "Female", "Product_Code": "SC", "Loan_Amount": "80000", "LTV": "85", "Resident_Type": "R",
     "Net_salary": "0", "Final_Tier": "01 Megapolis (A)", "Make_Code": "JUPITER",
     "Model_Description": "JUPITER ZX BSVI â€“ ZX DRUM", "Model_Variant": "110 CC", "PAST_LOANS_ACTIVE": "PAST_LOANS_ACTIVE"},
    {"ID": "T3", "Qualifications": "", "Employment_Type": "", "Age": "65", "Pincode": "",
     "Gender": "", "Product_Code": "", "Loan_Amount": "", "LTV": "", "Resident_Type": "",
     "Net_salary": "", "Final_Tier": "", "Make_Code": "", "Model_Description": "", "Model_Variant": "",
     "PAST_LOANS_ACTIVE": ""},
])

scored = basic_clean(nasty)          # Layer 1 (no row filter on test data!)
filled = full_prep.named_steps["domain"].transform(scored)
proba = models["Gradient boosting"].predict_proba(full_prep.transform(scored))[:, 1]
pd.DataFrame({"ID": nasty["ID"], "LTV (rebuilt if 0)": filled["LTV"].round(1),
              "tier_rank (filled if unknown)": filled["tier_rank"], "P(decline)": proba.round(3)})

,ID,LTV (rebuilt if 0),tier_rank (filled if unknown),P(decline)
0,T1,76.6,NaN,0.354
1,T2,85.0,1.0,0.178
2,T3,NaN,NaN,0.301


## 9. Save outputs

- `data/training_data_clean.csv`: training data after Layer 1 and the training-only filters. It is human-readable, for audit.
- `data/change_log.csv`: one row for **every cell we changed or removed**, with its original value, new value and reason. This is the audit trail for the panel.
- For the hidden test set, run: `basic_clean(test_df)` → `full_prep.transform(...)` → `model.predict_proba(...)`. Never re-fit on test data.

The change log covers Layer 1 and the business-aware fills (6a). The generic median fills in 6b happen inside the model pipeline; they are recorded by the "was missing" indicator columns instead.

In [18]:
# Apply the training-fitted business fills so the log shows the final LTV / tier values
audit = clean.copy()
filled_all = full_prep.named_steps["domain"].transform(audit.drop(columns=[TARGET, "ID", "label_conflict"]))
audit["LTV"] = filled_all["LTV"].values
audit["Tier_Rank_filled"] = filled_all["tier_rank"].values

NUMERIC = {"Age", "Loan_Amount", "LTV", "Net_salary"}
REASON_FIX = {"Model_Variant": "Standardised 'CC' spacing",
              "Model_Description": "Fixed garbled dash character (encoding error)",
              "LTV": "LTV was 0 (impossible); rebuilt from loan amount and typical bike price"}
REASON_FILL = {"Make_Code": "Blank; inferred from Product_Code (single make)",
               "Model_Variant": "Blank; inferred from Product_Code (single variant)",
               "LTV": "Blank; rebuilt from loan amount and typical bike price"}
REASON_BLANK = {"Net_salary": "Below Rs 1,000 for an earning applicant (typo); set to missing",
                "Age": "Outside plausible range; set to missing",
                "LTV": "Outside 0-100; set to missing",
                "Pincode": "Not a valid 6-digit pincode; set to missing"}

raw_by_id = raw.set_index("ID")
entries = []
for col in [c for c in raw.columns if c not in ("ID", "Decision")]:
    before = raw_by_id.loc[audit["ID"].astype(str), col].str.strip().replace("", np.nan).reset_index(drop=True)
    after = audit[col].reset_index(drop=True)
    if col in NUMERIC:
        b, a = pd.to_numeric(before, errors="coerce"), after.astype(float)
    else:
        b, a = before.astype(object), after.astype(object)
    changed = ~((b == a) | (b.isna() & a.isna()))
    for i in np.flatnonzero(changed):
        bi, ai = before.iloc[i], after.iloc[i]
        reason = (REASON_BLANK.get(col, "Invalid; set to missing") if pd.isna(ai)
                  else REASON_FILL.get(col, "Blank; filled") if pd.isna(bi)
                  else REASON_FIX.get(col, "Standardised"))
        entries.append((audit["ID"].iloc[i], col, bi, ai, reason))

# Tier filled from neighbouring pincodes (Final_Tier text stays blank; the model uses tier_rank)
tier_fill = audit["Final_Tier"].isna() & audit["Tier_Rank_filled"].notna()
for _, r in audit[tier_fill].iterrows():
    entries.append((r["ID"], "Final_Tier", np.nan, f"tier_rank={int(r['Tier_Rank_filled'])}",
                    "Blank; most common tier of same 3-digit pincode district"))

# Removed rows
dropped = sorted(set(raw["ID"]) - set(audit["ID"].astype(str)), key=lambda v: (v == "", v))
for rid in dropped:
    entries.append((rid if rid else "(blank)", "ALL", "row", "removed",
                    "Fully empty row" if rid == "" else "Exact duplicate of an earlier application"))

change_log = pd.DataFrame(entries, columns=["ID", "Column", "Original_Value", "Cleaned_Value", "Reason"])
change_log.to_csv("../data/change_log.csv", index=False)
print(f"Saved ../data/change_log.csv: {len(change_log):,} logged changes")
change_log.groupby(["Column", "Reason"]).size().to_frame("cells_changed")

Saved ../data/change_log.csv: 25,188 logged changes


cells_changed
Column            Reason                                                           
ALL               Exact duplicate of an earlier application                      13
                  Fully empty row                                                 1
Final_Tier        Blank; most common tier of same 3-digit pincode...             22
LTV               LTV was 0 (impossible); rebuilt from loan amoun...             65
Make_Code         Blank; inferred from Product_Code (single make)                 2
Model_Description Fixed garbled dash character (encoding error)                5027
Model_Variant     Blank; inferred from Product_Code (single variant)              2
                  Standardised 'CC' spacing                                   20011
Net_salary        Below Rs 1,000 for an earning applicant (typo);...             45

In [19]:
clean.to_csv("../data/training_data_clean.csv", index=False)
print("Saved ../data/training_data_clean.csv:", clean.shape)

Saved ../data/training_data_clean.csv: (108409, 28)


## Summary of cleaning decisions (for the panel)

1. **Read everything as text first.** This protects Pincode (a label, not a number) and the blank-space Age.
2. **Dropped (training only):** 1 fully empty row (which is also the missing-Decision row) and 13 duplicate applications.
3. **Kept but flagged:** 1,105 applicants over 60. 97% were approved, so this is real business history. Policy enforcement belongs in the chatbot rule layer.
4. **Salary:** zero is valid for students and non-earners. Values under Rs 1,000 for earners are typos, so we set them to missing. Extreme highs are capped at the 99.5th percentile from training data.
5. **LTV = 0** (65 rows) is impossible, so we rebuild it from the loan amount and that bike model's typical price.
6. **Missing values:** a blank tier is taken from neighbouring pincodes, and a blank make from the product (moped / e-bike). Everything else gets the training median, or a "Missing" category, plus a flag, because missingness itself can be a signal.
7. **Text fixes:** garbled dash characters and `125CC`/`125 CC` spelling.
8. **Pincode** is reduced to zone and district, and target-encoded with smoothing. This handles 14,000 codes and unseen ones.
9. **Unseen categories** in the test set fall into an "infrequent" bucket or the overall average. Nothing crashes, as section 8 shows.
10. **Imbalance** is handled with class weights and AUC metrics, not by resampling the data.
11. **No leakage:** every learned number (medians, caps, encodings, bike prices, tier lookups) comes from the training split only.
12. **Gender is excluded** from the model. It shows no difference in decline rate (chi-square test) and is a fair-lending risk. It is kept in the data only for fairness audits.
13. **Label conflicts:** 6 rows have identical details but opposite decisions. We keep and flag them as evidence of information missing from our data, such as a bureau score.
14. **Audit trail:** every changed or removed cell is listed in `data/change_log.csv`, alongside the decline-rate tables in section 4b.